In [149]:
import pandas as pd
import numpy as np
import re

In [150]:
df = pd.read_csv('smartphones.csv')
df.head(2)

,Model,Price,Rating,Spec Score,Sim,Processor,RAM,Battery,Display,Camera,Card,OS
0,Samsung Galaxy S25 Ultra,"₹1,07,550",4.15,93,"Dual Sim, 3G, 4G, 5G, VoLTE, Vo5G, Wi-Fi, NFC","Snapdragon 8 Elite for Galaxy, Octa Core, 4.47...","12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 45W Fast Charging,"6.9 inches, 1440 x 3120 px, 120 Hz Display wit...",200 MP Quad Rear & 12 MP Front Camera,Memory Card Not Supported,Android v15
1,Samsung Galaxy S25 FE,"₹59,999",4.70,86,"Dual Sim, 3G, 4G, 5G, VoLTE, Vo5G, Wi-Fi, NFC","Exynos 2400, Deca Core, 3.2 GHz Processor","8 GB RAM, 128 GB inbuilt",4900 mAh Battery with 45W Fast Charging,"6.7 inches, 1080 x 2340 px, 120 Hz Display wit...",50 MP + 12 MP + 8 MP Triple Rear & 12 MP Front...,Memory Card Not Supported,Android v16


### symbol use
1. consistency - CON
2. validity - V
3. accuracy - A
4. completness - COM

### quality issue
  1. `model` - some brands are written differently OPPO and Oppo in model column - CON (done)
  2. `model` - some brands have (ram + rom) mention in there name - CON (done)
  3. `price` - has unneccesary '₹' symbol - V (done)
  4. `price` - has ',' between numbers - V (done)
  5. `price` - some Phone price has very low -  A
  6. `camera and os` - has missing value - COM
  7. `processor` - has some missing value - COM
  8. `row 421` - it data has shifted - V
  9. `RAM` - inccorect value - V
  10. `Battery` - incorrect value -V
  11. `Display` - some has incorrect value - V
  12. `Display` - sometime refresh rate is not there - COM
  13. `Camera` - has some incorrect vlues - V
  14. certain phones are data scattered - V
  15. `camera` - some has incorrect value - V
  17. Bacically all columnn has some incorrect values because of data shifted - V
  18. `camera, card and os` - has missing values - COM
  19. datatype of `price` is incorrect - V (done)

### Tidiness issues
  1. `Sim` - can be split into 4 cols 4g, 5g, has NFC, and has IR_blaster (done)
  2. `RAM` - can be split into two cols RAM and ROM
  3. `Processor` - can be split into processor_name, cores, cpu_speed (done)
  4. `Battery` - can be split into capacity and carging_watt
  5. `display` - can be split into display_size, resolution_width, resolution_height and frequency
  6. `camera` - can be split into fron and rear camera
  7. `card` - can be split inot supported, extended_upto
  8. `os` - some os name is given like lollipop - CON

## basic check

In [151]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Model       1020 non-null   object 
 1   Price       1020 non-null   object 
 2   Rating      1020 non-null   float64
 3   Spec Score  1020 non-null   int64  
 4   Sim         1020 non-null   object 
 5   Processor   1020 non-null   object 
 6   RAM         1020 non-null   object 
 7   Battery     1020 non-null   object 
 8   Display     1020 non-null   object 
 9   Camera      1017 non-null   object 
 10  Card        1012 non-null   object 
 11  OS          998 non-null    object 
dtypes: float64(1), int64(1), object(10)
memory usage: 95.8+ KB


In [152]:
df.describe()

,Rating,Spec Score
count,1020.000000,1020.000000
mean,4.373284,75.295098
std,0.232956,18.059170
min,3.900000,6.000000
25%,4.150000,74.000000
50%,4.375000,80.000000
75%,4.600000,84.000000
max,4.750000,97.000000


In [153]:
df.duplicated().sum()

np.int64(0)

In [154]:
df.sample(2)

,Model,Price,Rating,Spec Score,Sim,Processor,RAM,Battery,Display,Camera,Card,OS
331,Poco C85 5G,"₹11,999",4.7,75,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Dimensity 6300, Octa Core, 2.4 GHz Processor","4 GB RAM, 128 GB inbuilt",6000 mAh Battery with 33W Fast Charging,"6.9 inches, 720 x 1600 px, 120 Hz Display with...",50 MP Dual Rear & 8 MP Front Camera,"Memory Card (Hybrid), upto 1 TB",Android v15
999,GFive U101,₹549,4.2,11,Dual Sim,"Spreadtrum, 1.2 GHz Processor","32 MB RAM, 32 MB inbuilt",1000 mAh Battery,"1.8 inches, 320 x 480 px Display",0.3 MP Rear Camera,v30,Bluetooth


## cleaning, removing incorrect value shifting data from the column processor ram, battery, display, camera, card and os

#### Removing Non smartphone

In [155]:
df.columns

Index(['Model', 'Price', 'Rating', 'Spec Score', 'Sim', 'Processor', 'RAM',
       'Battery', 'Display', 'Camera', 'Card', 'OS'],
      dtype='object')

In [156]:
# convert smartphone prize into int so i can check
# Remove ₹ and , then convert to a numeric type
df['Price'] = df['Price'].str.replace(r'[₹,]', '', regex=True)

# Important: Convert the column to float or int so you can do math
df['Price'] = pd.to_numeric(df['Price'])

In [157]:
# df[(df['Price'] >= 5500) & (df['Price'] <= 6000)]
# by hit and trial we get the prize threshold to 5500 rupee
# so we filter out the phone which are below this range
df = df[df['Price'] >= 5500]
df.shape

(949, 12)

#### now shifting data

In [158]:
# double shift
# for processor - > Battery to all
mask = df['Processor'].str.contains('mAh', case=False, na=False)

# 2. Define columns from index 5 (Processor) to the end
cols_to_fix = df.columns[5:]

print("total number of items to fix ", mask.sum())
df.loc[mask, cols_to_fix] = df.loc[mask, cols_to_fix].shift(2, axis=1)

total number of items to fix  0


In [159]:
# single right shift
# for processor - > RAM to all
mask = df['Processor'].str.contains('inbuilt|RAM', case=False, na=False)

# 2. Define columns from index 5 (Processor) to the end
cols_to_fix = df.columns[5:]

print("total number of items to fix ", mask.sum())
df.loc[mask, cols_to_fix] = df.loc[mask, cols_to_fix].shift(1, axis=1)

total number of items to fix  0


In [160]:
# single left
# for Ram -> processor
mask = df['RAM'].str.contains('Processor|unisoc|asr', case=False, na=False)
# 2. Define columns from index 5 (Processor) to the end
cols_to_fix = df.columns[6:]

print("total number of items to fix ", mask.sum())
df.loc[mask, cols_to_fix] = df.loc[mask, cols_to_fix].shift(-1, axis=1)

total number of items to fix  0


In [161]:
mask = df['RAM'].str.contains('mAh', case=False, na=False)
# 2. Define columns from index 5 (Processor) to the end
cols_to_fix = df.columns[6:]

print("total number of items to fix ", mask.sum())
df.loc[mask, cols_to_fix] = df.loc[mask, cols_to_fix].shift(1, axis=1)
# df[mask]

total number of items to fix  0


In [162]:
mask = df['Camera'].str.contains('memory', case=False, na=False)
# 2. Define columns from index 5 (Processor) to the end
cols_to_fix = df.columns[9:]

print("total number of items to fix ", mask.sum())
df.loc[mask, cols_to_fix] = df.loc[mask, cols_to_fix].shift(1, axis=1)
# df[mask]

total number of items to fix  1


In [163]:
mask = df['Card'].str.contains('android|ios|Symbian', case=False, na=False)
# 2. Define columns from index 5 (Processor) to the end
cols_to_fix = df.columns[10:]

print("total number of items to fix ", mask.sum())
df.loc[mask, cols_to_fix] = df.loc[mask, cols_to_fix].shift(1, axis=1)
# df[mask]

total number of items to fix  260


In [164]:
# extracting all the incorrect values of processor and we didn't reset index so we can remove from the original
df_out_processor = df[df['Processor'].astype(str).str.contains('Wifi|inbuilt|RAM|Battery', case=False, na=False)]

# removing item from the main df and resetting the index now do same for other also\
df = df.drop(df_out_processor.index).reset_index(drop=True)

# same for ram
df_out_ram = df[~df['RAM'].astype(str).str.contains('inbuilt|RAM', case=False, na=False)]
df = df.drop(df_out_ram.index).reset_index(drop=True)

# for battery
df_out_battery = df[~df['Battery'].astype(str).str.contains('mAh|Fast|Charging', case=False, na=False)]
df = df.drop(df_out_battery.index).reset_index(drop=True)

# for display
df_out_display = df[~df['Display'].astype(str).str.contains('inches|px|Display|Hz', case=False, na=False)]
df = df.drop(df_out_display.index).reset_index(drop=True)

# for camera
# ther are many rows where there is written foldable in camera and the data of camera is shifted to card so we do this first
# 1. Identify the rows that need fixing
idx = df[~df['Camera'].astype(str).str.contains('MP|Camera|Rear', case=False, na=False)].index

# 2. Move 'Card' to 'Camera' ONLY for those indices
df.loc[idx, 'Camera'] = df.loc[idx, 'Card']

# 3. Now explicitly clear the 'Card' column for those same indices
df.loc[idx, 'Card'] = np.nan

df_out_camera = df[~df['Camera'].astype(str).str.contains('MP|Camera|Rear', case=False, na=False)]
df = df.drop(df_out_camera.index).reset_index(drop=True)



# for card
idx = df[df['Card'].astype(str).str.contains('Android|FM|Radio', case=False, na=False)].index
df.loc[idx, 'OS'] = df.loc[idx, 'Card']
# 3. Now explicitly clear the 'Card' column for those same indices
df.loc[idx, 'Card'] = np.nan


# We will keep only the rows that mention "GB", "TB", or "upto", and set everything else (including the "Not Supported" text) to a real NaN.
df.loc[~df['Card'].astype(str).str.contains('GB|TB|upto|memory', case=False, na=False), 'Card'] = np.nan


#for os
idx = df[(df['OS'].str.contains('Memory', case=False, na=False))].index
df.loc[idx, 'Card'] = df.loc[idx, 'OS']
df.loc[idx, 'OS'] = np.nan
df.loc[~df['OS'].str.contains('android|ios|Symbian', case=False, na=False), 'OS'] = np.nan


# concatinate all the out rows
# Combine the two dataframes

df_out = pd.concat([df_out_processor, df_out_ram,df_out_battery,df_out_display,df_out_camera], axis=0)

# Reset the index after merging so it's a clean 0 to N sequence
df_out = df_out.reset_index(drop=True)

# df.to_csv("new_clean.csv",index=False)

In [165]:
df.shape

(948, 12)

In [166]:
df_out

,Model,Price,Rating,Spec Score,Sim,Processor,RAM,Battery,Display,Camera,Card,OS
0,Apple iPhone 12 (128GB),54900,4.55,74,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt","6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,None,Memory Card Not Supported,iOS v14


## Price

In [167]:
# # Remove ₹ and , then convert to a numeric type
# df['Price'] = df['Price'].str.replace(r'[₹,]', '', regex=True)

# # Important: Convert the column to float or int so you can do math
# df['Price'] = pd.to_numeric(df['Price'])

# df.sample()

In [168]:
df.describe()

,Price,Rating,Spec Score
count,948.000000,948.000000,948.000000
mean,31302.254219,4.375316,79.755274
std,30793.836171,0.232968,7.721473
min,5599.000000,3.900000,38.000000
25%,13989.750000,4.150000,76.000000
50%,21627.000000,4.400000,81.000000
75%,33999.000000,4.600000,84.000000
max,229900.000000,4.750000,97.000000


## Model

In [169]:
# convert it to a small letter so the OPPO, Oppo and oppo three are same company
df['Model'] = df['Model'].astype(str).str.lower()

In [170]:
# before cleanup
print(df["Model"][773:777])
# remove (8gb + 12 ram) this type from the Model column
df['Model'] = df['Model'].astype(str).str.replace(r'\s*\(.*\)', '', regex=True).str.strip()

# Clean up any trailing/leading (front/back) spaces just in case
df['Model'] = df['Model'].str.strip()

# after cleanup
print(df["Model"][773:777])

773              samsung galaxy a16 5g (8gb ram + 256gb)
774                  tecno spark 30c 5g (4gb ram + 64gb)
775                     tecno pop 9 5g (4gb ram + 128gb)
776    samsung galaxy m15 5g prime edition (6gb ram +...
Name: Model, dtype: object
773                  samsung galaxy a16 5g
774                     tecno spark 30c 5g
775                         tecno pop 9 5g
776    samsung galaxy m15 5g prime edition
Name: Model, dtype: object


In [171]:
df.head()

,Model,Price,Rating,Spec Score,Sim,Processor,RAM,Battery,Display,Camera,Card,OS
0,samsung galaxy s25 ultra,107550,4.15,93,"Dual Sim, 3G, 4G, 5G, VoLTE, Vo5G, Wi-Fi, NFC","Snapdragon 8 Elite for Galaxy, Octa Core, 4.47...","12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 45W Fast Charging,"6.9 inches, 1440 x 3120 px, 120 Hz Display wit...",200 MP Quad Rear & 12 MP Front Camera,Memory Card Not Supported,Android v15
1,samsung galaxy s25 fe,59999,4.70,86,"Dual Sim, 3G, 4G, 5G, VoLTE, Vo5G, Wi-Fi, NFC","Exynos 2400, Deca Core, 3.2 GHz Processor","8 GB RAM, 128 GB inbuilt",4900 mAh Battery with 45W Fast Charging,"6.7 inches, 1080 x 2340 px, 120 Hz Display wit...",50 MP + 12 MP + 8 MP Triple Rear & 12 MP Front...,Memory Card Not Supported,Android v16
2,realme 16 pro 5g,31999,4.05,77,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Dimensity 7300 Max, Octa Core Processor","8 GB RAM, 128 GB inbuilt",7000 mAh Battery with 80W Fast Charging,"6.78 inches, 1272 x 2772 px, 144 Hz Display wi...",200 MP + 8 MP Dual Rear & 50 MP Front Camera,Memory Card Not Supported,Android v16
3,realme 16 pro plus 5g,39999,4.10,90,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Snapdragon 7 Gen4, Octa Core, 2.8 GHz Processor","8 GB RAM, 128 GB inbuilt",7000 mAh Battery with 80W Fast Charging,"6.8 inches, 1280 x 2800 px, 144 Hz Display wit...",200 MP + 50 MP + 8 MP Triple Rear & 50 MP Fron...,NaN,Android v16
4,oneplus nord ce 5 5g,24998,4.10,83,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, IR Blaster","Dimensity 8350 Apex, Octa Core, 3.35 GHz Proce...","8 GB RAM, 128 GB inbuilt",7100 mAh Battery with 80W Fast Charging,"6.77 inches, 1080 x 2392 px, 120 Hz Display wi...",50 MP + 8 MP Dual Rear & 16 MP Front Camera,"Memory Card (Hybrid), upto 1 TB",Android v15


## sim

In [172]:
# This creates a Boolean column (True/False)
df['5G'] = df['Sim'].str.contains('5G', case=False, na=False)
df['4G'] = df['Sim'].str.contains('4G', case=False, na=False)
df['NFC'] = df['Sim'].str.contains('NFC', case=False, na=False)
df['IR'] = df['Sim'].str.contains('IR', case=False, na=False)
# Drop the Sim column
df.drop('Sim', axis=1, inplace=True)

In [173]:
# Filter rows where NONE of the specified columns are True
len(df[~df[['5G', '4G', 'NFC', 'IR']].any(axis=1)])

0

## Processor

In [174]:
# processor speed
df['Processor_Speed'] = df['Processor'].str.extract(
    r'(\d+\.?\d*\s?[KMG]Hz)\s+Processor', 
    flags=re.IGNORECASE
)
# num cores
df['Num_Cores'] = df['Processor'].str.extract(r'(\w+\s+Core)', flags=re.IGNORECASE)

# processor name
df['Processor_Name'] = df['Processor'].str.replace(r'\(|\)|Processor', '', flags=re.IGNORECASE, regex=True)
df['Processor_Name'] = df['Processor_Name'].str.replace(r'\w+\s+Core', '',flags=re.IGNORECASE, regex=True)
df['Processor_Name'] = df['Processor_Name'].str.replace(r'\d+\.?\d*\s?[KMG]Hz', '', flags=re.IGNORECASE, regex=True)
df['Processor_Name'] = df['Processor_Name'].str.strip(', ')

# drop the processor column
# Drop the Sim column
df.drop('Processor', axis=1, inplace=True)

In [175]:
# converting to cores in numeric value 
# Create a mapping dictionary for cores
core_map = {
    'Single Core': 1,
    'Dual Core': 2,
    'Quad Core': 4,
    'Hexa Core': 6,
    'Octa Core': 8,
    'Nine Core': 9,
    'Deca Core': 10
}

# Apply the map (NaN values will stay NaN automatically)
df['Num_Cores'] = df['Num_Cores'].map(core_map).astype(float)

In [176]:
def convert_speed_to_ghz(text):
    if pd.isna(text): return np.nan
    
    # Extract number and unit
    match = re.search(r'(\d+(?:\.\d+)?)\s*([a-z]+)', str(text), re.IGNORECASE)
    if match:
        val = float(match.group(1))
        unit = match.group(2).lower()
        
        if unit == 'ghz':
            return val
        elif unit == 'mhz':
            return val / 1000
        elif unit == 'khz':
            return val / 1000000
    return np.nan

df['Processor_Speed'] = df['Processor_Speed'].apply(convert_speed_to_ghz)

## RAM

In [177]:
df['ram'] = df['RAM'].str.extract(r'(\d+\s*[MGT]B)\s*RAM',flags=re.IGNORECASE)
df['rom'] = df['RAM'].str.extract(r'(\d+\s*[MGT]B)\s*inbuilt',flags=re.IGNORECASE)
df.drop('RAM', axis=1, inplace=True)

In [178]:
def convert_column_to_gb(series):
    # 1. Extract the number and the unit separately
    # This regex looks for a number (decimal or int) and the unit (MB, GB, or TB)
    extracted = series.str.extract(r'(\d+(?:\.\d+)?)\s*([MGT]B)', expand=True, flags=re.IGNORECASE)
    
    # Rename columns for clarity: 0 is the value, 1 is the unit
    nums = pd.to_numeric(extracted[0])
    units = extracted[1].str.upper()
    
    # 2. Define the conversion logic
    # Default to 1 (for GB), divide by 1024 (for MB), multiply by 1024 (for TB)
    factors = np.select(
        [units == 'MB', units == 'GB', units == 'TB'],
        [1/1024, 1, 1024],
        default=np.nan
    )
    
    return nums * factors

# Apply to your existing columns
df['ram'] = convert_column_to_gb(df['ram'])
df['rom'] = convert_column_to_gb(df['rom'])

In [179]:
df.describe()

,Price,Rating,Spec Score,Processor_Speed,Num_Cores,ram,rom
count,948.000000,948.000000,948.000000,907.000000,946.000000,945.000000,948.000000
mean,31302.254219,4.375316,79.755274,2.657755,7.894292,8.150265,210.928270
std,30793.836171,0.232968,7.721473,0.597426,0.739962,3.079914,153.926103
min,5599.000000,3.900000,38.000000,0.001300,1.000000,2.000000,8.000000
25%,13989.750000,4.150000,76.000000,2.400000,8.000000,6.000000,128.000000
50%,21627.000000,4.400000,81.000000,2.500000,8.000000,8.000000,128.000000
75%,33999.000000,4.600000,84.000000,3.000000,8.000000,8.000000,256.000000
max,229900.000000,4.750000,97.000000,4.600000,10.000000,24.000000,2048.000000


## Battery

In [180]:
# 1. Extract the battery capacity (e.g., 6000 from '6000 mAh')
# Using (\d+(?:\.\d+)?) to catch decimals like 4500.5 just in case
df['battery_capacity'] = df['Battery'].str.extract(r'(\d+(?:\.\d+)?)\s*mAh', flags=re.IGNORECASE).astype(float)

# 2. Extract the charging wattage (e.g., 20 from '20W')
df['charging_wattage'] = df['Battery'].str.extract(r'(\d+(?:\.\d+)?)\s*W', flags=re.IGNORECASE).astype(float)

# 3. Drop the original 'Battery' column
df.drop('Battery', axis=1, inplace=True)

In [181]:
df.describe()

,Price,Rating,Spec Score,Processor_Speed,Num_Cores,ram,rom,battery_capacity,charging_wattage
count,948.000000,948.000000,948.000000,907.000000,946.000000,945.000000,948.000000,948.000000,875.000000
mean,31302.254219,4.375316,79.755274,2.657755,7.894292,8.150265,210.928270,5322.856540,50.770857
std,30793.836171,0.232968,7.721473,0.597426,0.739962,3.079914,153.926103,866.723324,29.115652
min,5599.000000,3.900000,38.000000,0.001300,1.000000,2.000000,8.000000,2000.000000,10.000000
25%,13989.750000,4.150000,76.000000,2.400000,8.000000,6.000000,128.000000,5000.000000,25.000000
50%,21627.000000,4.400000,81.000000,2.500000,8.000000,8.000000,128.000000,5000.000000,45.000000
75%,33999.000000,4.600000,84.000000,3.000000,8.000000,8.000000,256.000000,6000.000000,80.000000
max,229900.000000,4.750000,97.000000,4.600000,10.000000,24.000000,2048.000000,7550.000000,150.000000


## Display

In [182]:
# 1. Extract Screen Size (Inches)
# Pattern: Looks for a number (inc. decimals) followed by 'inch' or 'inches'
df['inch'] = df['Display'].str.extract(r'(\d+(?:\.\d+)?)\s*inch', flags=re.IGNORECASE).astype(float)

# 2. Extract Resolution Width and Height
# Pattern: Looks for 'number x number' followed by 'px'
# We use .extractall or a multi-group extract
resolution = df['Display'].str.extract(r'(\d+)\s*x\s*(\d+)\s*px', flags=re.IGNORECASE)
df['resolution_width'] = resolution[0].astype(float)
df['resolution_height'] = resolution[1].astype(float)

# 3. Extract Refresh Rate (Hz)
# Pattern: Looks for digits followed by 'Hz'
df['refresh_rate'] = df['Display'].str.extract(r'(\d+)\s*Hz', flags=re.IGNORECASE).astype(float)

# 4. Drop the original column
df.drop('Display', axis=1, inplace=True)

In [183]:
df.describe()

,Price,Rating,Spec Score,Processor_Speed,Num_Cores,ram,rom,battery_capacity,charging_wattage,inch,resolution_width,resolution_height,refresh_rate
count,948.000000,948.000000,948.000000,907.000000,946.000000,945.000000,948.000000,948.000000,875.000000,948.000000,947.000000,947.000000,860.000000
mean,31302.254219,4.375316,79.755274,2.657755,7.894292,8.150265,210.928270,5322.856540,50.770857,6.671677,1083.447730,2317.450898,118.193023
std,30793.836171,0.232968,7.721473,0.597426,0.739962,3.079914,153.926103,866.723324,29.115652,0.289105,254.644033,473.781667,14.188403
min,5599.000000,3.900000,38.000000,0.001300,1.000000,2.000000,8.000000,2000.000000,10.000000,3.100000,128.000000,160.000000,90.000000
25%,13989.750000,4.150000,76.000000,2.400000,8.000000,6.000000,128.000000,5000.000000,25.000000,6.670000,1080.000000,2340.000000,120.000000
50%,21627.000000,4.400000,81.000000,2.500000,8.000000,8.000000,128.000000,5000.000000,45.000000,6.700000,1080.000000,2400.000000,120.000000
75%,33999.000000,4.600000,84.000000,3.000000,8.000000,8.000000,256.000000,6000.000000,80.000000,6.770000,1220.000000,2640.000000,120.000000
max,229900.000000,4.750000,97.000000,4.600000,10.000000,24.000000,2048.000000,7550.000000,150.000000,8.030000,2460.000000,3216.000000,165.000000


## Camera

In [184]:
# 1. Primary Rear Camera
# Simply grabs the very first number in the string (which is always Rear in your format)
df['primary_rear_camera'] = df['Camera'].str.extract(r'^(\d+(?:\.\d+)?)\s*MP', flags=re.IGNORECASE).astype(float)

# Counts how many 'MP' occur before the word 'Rear' to get the number of lenses
df['rear_camera_count'] = df['Camera'].str.split('Rear').str[0].str.count('MP', flags=re.IGNORECASE)

# 2. Primary Front Camera
# Logic: Split by '&', take the second part (index 1), then grab the first number in that part
df['primary_front_camera'] = df['Camera'].str.split('&').str[1].str.extract(r'(\d+(?:\.\d+)?)\s*MP', flags=re.IGNORECASE).astype(float)

# 3. Clean up
df.drop('Camera', axis=1, inplace=True)

In [185]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 948 entries, 0 to 947
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Model                 948 non-null    object 
 1   Price                 948 non-null    int64  
 2   Rating                948 non-null    float64
 3   Spec Score            948 non-null    int64  
 4   Card                  676 non-null    object 
 5   OS                    925 non-null    object 
 6   5G                    948 non-null    bool   
 7   4G                    948 non-null    bool   
 8   NFC                   948 non-null    bool   
 9   IR                    948 non-null    bool   
 10  Processor_Speed       907 non-null    float64
 11  Num_Cores             946 non-null    float64
 12  Processor_Name        948 non-null    object 
 13  ram                   945 non-null    float64
 14  rom                   948 non-null    float64
 15  battery_capacity      9

## Card

In [186]:
# 1. Clean up "card_supported"
# This looks for any mention of 'Memory Card' OR 'Hybrid' but EXCLUDES 'Not Supported'
df['card_supported'] = df['Card'].str.contains(r'Memory Card|Hybrid', case=False, na=np.nan)

# Explicitly set False if "Not Supported" is in the text
df.loc[df['Card'].str.contains('Not Supported', case=False, na=False), 'card_supported'] = False

# 2. Refined Capacity Extraction
def extract_card_gb(text):
    if pd.isna(text): return np.nan
    
    # regex matches: numbers followed by optional space and GB or TB
    # We look for the number anywhere in the string that is near GB or TB
    match = re.search(r'(\d+)\s*(GB|TB)', str(text), re.IGNORECASE)
    
    if match:
        val = float(match.group(1))
        unit = match.group(2).upper()
        if unit == 'TB':
            return val * 1024
        return val
    return np.nan

df['max_card_capacity_GB'] = df['Card'].apply(extract_card_gb)

# 3. Final Drop
df.drop('Card', axis=1, inplace=True)

In [187]:
df.head()

,Model,Price,Rating,Spec Score,OS,5G,4G,NFC,IR,Processor_Speed,...,charging_wattage,inch,resolution_width,resolution_height,refresh_rate,primary_rear_camera,rear_camera_count,primary_front_camera,card_supported,max_card_capacity_GB
0,samsung galaxy s25 ultra,107550,4.15,93,Android v15,True,True,True,False,4.47,...,45.0,6.90,1440.0,3120.0,120.0,200.0,1,12.0,False,NaN
1,samsung galaxy s25 fe,59999,4.70,86,Android v16,True,True,True,False,3.20,...,45.0,6.70,1080.0,2340.0,120.0,50.0,3,12.0,False,NaN
2,realme 16 pro 5g,31999,4.05,77,Android v16,True,True,False,False,NaN,...,80.0,6.78,1272.0,2772.0,144.0,200.0,2,50.0,False,NaN
3,realme 16 pro plus 5g,39999,4.10,90,Android v16,True,True,True,True,2.80,...,80.0,6.80,1280.0,2800.0,144.0,200.0,3,50.0,NaN,NaN
4,oneplus nord ce 5 5g,24998,4.10,83,Android v15,True,True,False,True,3.35,...,80.0,6.77,1080.0,2392.0,120.0,50.0,2,16.0,True,1024.0


## final save and discription

In [189]:
df.shape

(948, 25)

In [190]:
df

,Model,Price,Rating,Spec Score,OS,5G,4G,NFC,IR,Processor_Speed,...,charging_wattage,inch,resolution_width,resolution_height,refresh_rate,primary_rear_camera,rear_camera_count,primary_front_camera,card_supported,max_card_capacity_GB
0,samsung galaxy s25 ultra,107550,4.15,93,Android v15,True,True,True,False,4.47,...,45.0,6.90,1440.0,3120.0,120.0,200.0,1,12.0,False,NaN
1,samsung galaxy s25 fe,59999,4.70,86,Android v16,True,True,True,False,3.20,...,45.0,6.70,1080.0,2340.0,120.0,50.0,3,12.0,False,NaN
2,realme 16 pro 5g,31999,4.05,77,Android v16,True,True,False,False,NaN,...,80.0,6.78,1272.0,2772.0,144.0,200.0,2,50.0,False,NaN
3,realme 16 pro plus 5g,39999,4.10,90,Android v16,True,True,True,True,2.80,...,80.0,6.80,1280.0,2800.0,144.0,200.0,3,50.0,NaN,NaN
4,oneplus nord ce 5 5g,24998,4.10,83,Android v15,True,True,False,True,3.35,...,80.0,6.77,1080.0,2392.0,120.0,50.0,2,16.0,True,1024.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
943,xiaomi redmi note 5 pro,9239,4.20,69,Android v7.1.2 (Nougat),False,True,False,True,1.80,...,NaN,5.99,1080.0,2160.0,NaN,12.0,2,20.0,True,128.0
944,xiaomi redmi 5a,6299,3.95,50,Android v7.1 (Nougat),False,True,False,False,1.40,...,NaN,5.00,720.0,1280.0,NaN,13.0,1,5.0,True,NaN
945,apple iphone 8 plus,49900,4.30,67,iOS v11,False,True,True,False,2.39,...,NaN,5.50,1080.0,1920.0,NaN,12.0,2,7.0,False,NaN
946,xiaomi redmi 4,6999,4.20,60,Android v6.0.1 (Marshmallow),False,True,False,True,1.40,...,NaN,5.00,720.0,1280.0,NaN,13.0,1,5.0,True,128.0


In [191]:
df.to_csv("Full_clean_seperated_data.csv",index=False)

In [192]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 948 entries, 0 to 947
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Model                 948 non-null    object 
 1   Price                 948 non-null    int64  
 2   Rating                948 non-null    float64
 3   Spec Score            948 non-null    int64  
 4   OS                    925 non-null    object 
 5   5G                    948 non-null    bool   
 6   4G                    948 non-null    bool   
 7   NFC                   948 non-null    bool   
 8   IR                    948 non-null    bool   
 9   Processor_Speed       907 non-null    float64
 10  Num_Cores             946 non-null    float64
 11  Processor_Name        948 non-null    object 
 12  ram                   945 non-null    float64
 13  rom                   948 non-null    float64
 14  battery_capacity      948 non-null    float64
 15  charging_wattage      8

In [193]:
df.describe()

,Price,Rating,Spec Score,Processor_Speed,Num_Cores,ram,rom,battery_capacity,charging_wattage,inch,resolution_width,resolution_height,refresh_rate,primary_rear_camera,rear_camera_count,primary_front_camera,max_card_capacity_GB
count,948.000000,948.000000,948.000000,907.000000,946.000000,945.000000,948.000000,948.000000,875.000000,948.000000,947.000000,947.000000,860.000000,948.000000,948.000000,945.000000,374.000000
mean,31302.254219,4.375316,79.755274,2.657755,7.894292,8.150265,210.928270,5322.856540,50.770857,6.671677,1083.447730,2317.450898,118.193023,56.782911,2.199367,20.189630,1332.791444
std,30793.836171,0.232968,7.721473,0.597426,0.739962,3.079914,153.926103,866.723324,29.115652,0.289105,254.644033,473.781667,14.188403,34.043412,0.734804,13.989469,850.023973
min,5599.000000,3.900000,38.000000,0.001300,1.000000,2.000000,8.000000,2000.000000,10.000000,3.100000,128.000000,160.000000,90.000000,5.000000,1.000000,2.000000,32.000000
25%,13989.750000,4.150000,76.000000,2.400000,8.000000,6.000000,128.000000,5000.000000,25.000000,6.670000,1080.000000,2340.000000,120.000000,50.000000,2.000000,8.000000,1024.000000
50%,21627.000000,4.400000,81.000000,2.500000,8.000000,8.000000,128.000000,5000.000000,45.000000,6.700000,1080.000000,2400.000000,120.000000,50.000000,2.000000,16.000000,1024.000000
75%,33999.000000,4.600000,84.000000,3.000000,8.000000,8.000000,256.000000,6000.000000,80.000000,6.770000,1220.000000,2640.000000,120.000000,50.000000,3.000000,32.000000,2048.000000
max,229900.000000,4.750000,97.000000,4.600000,10.000000,24.000000,2048.000000,7550.000000,150.000000,8.030000,2460.000000,3216.000000,165.000000,200.000000,3.000000,50.000000,5120.000000
